# 04 - Phân tích Lỗi và Kết hợp Mô hình (Model Blending)

Trong bài học này, chúng ta tìm hiểu kỹ thuật kết hợp mô hình (Blending):
- Khảo sát mô hình **EfficientNet-B2** ($288 \times 288$) và lưu ý về các yếu tố thay đổi đồng thời.
- Căn chỉnh dự đoán validation giữa mô hình **Native (DenseNet-121)** và **B2 (EfficientNet-B2)** trên Fold 0.
- Phân tích ma trận chồng chéo lỗi 4 góc (4-way error overlap).
- Khảo sát các mẫu lỗi cụ thể kèm dự đoán của cả hai mô hình và nhãn thực tế.
- Đánh giá mô hình Blending cố định $50\% \text{ B2} + 50\% \text{ Native}$ và đếm số lỗi sửa được vs số lỗi mới sinh ra.

In [ ]:
from pathlib import Path
import os
import sys

def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import torch
import numpy as np
import pandas as pd

from kmd.core import PACKAGE, read_csv, split_fold, metric, read_json
from kmd.config import Config
from kmd.pipeline import prepare_development, load_session, train_cnn_fold, aligned_predictions, blend

RUN_ID = 'lesson_session'
print(f"Phiên làm việc: {RUN_ID}")

## 1. EfficientNet-B2 và Các yếu tố thay đổi đồng thời

Cấu hình `b2.json` khác với `native.json` ở nhiều yếu tố đồng thời:
- Backbone: EfficientNet-B2 vs DenseNet-121.
- Kích thước ảnh: $288 \times 288$ vs $224 \times 224$.
- Chiến lược nhìn: `center60` vs 4 crops.
- Số epoch: tối đa 48 với early stopping vs 19 cố định.

Do đó, sự khác biệt hiệu năng giữa B2 và Native phản ánh tổng hòa của nhiều yếu tố thiết kế chứ không phải một biến đối chứng đơn lẻ.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Bước huấn luyện CNN cần CUDA. Bạn có thể chạy nhánh LR ở bài 01 và 05 trên CPU.")

data_root_env = os.environ.get('DATA_ROOT')
data_root = Path(data_root_env or TASK_ROOT / 'data/train').expanduser().resolve()
if not (data_root / 'pairs.csv').is_file():
    raise FileNotFoundError(f'Thiếu dữ liệu train: {data_root / "pairs.csv"}. Xem README để đặt DATA_ROOT.')

if (data_root / 'pairs.csv').is_file() and torch.cuda.is_available():
    dev_frame = prepare_development(data_root)
    tr_fold0, va_fold0 = split_fold(dev_frame, fold=0)
    session_dir = load_session(RUN_ID, data_root)
    
    # Kiểm tra tiền đề: Native Fold 0 phải tồn tại từ notebook 03
    native_folder = session_dir / 'native_fold0'
    if not (native_folder / 'best.pt').is_file():
        raise FileNotFoundError(
            f"Chưa tìm thấy kết quả Native Fold 0 tại {native_folder}. "
            "Vui lòng chạy notebook 03_native_and_resampling.ipynb trước."
        )
        
    # File tồn tại chưa đủ: wrapper kiểm tra cấu hình, metadata và hash trước khi dùng lại.
    native_folder, _ = train_cnn_fold('native', dev_frame, data_root, session_dir, fold=0)

    # Huấn luyện / Nạp B2 Fold 0
    folder_b2, res_b2 = train_cnn_fold('b2', dev_frame, data_root, session_dir, fold=0)
    
    pred_nat = read_csv(native_folder / 'development.csv').rename(columns={'y': 'fake_position', 'fold': 'inner_fold'})
    pred_b2 = read_csv(folder_b2 / 'development.csv').rename(columns={'y': 'fake_position', 'fold': 'inner_fold'})
    
    pred_nat = aligned_predictions(pred_nat, va_fold0)
    pred_b2 = aligned_predictions(pred_b2, va_fold0)
    print("Đã căn chỉnh dự đoán của Native và B2 trên các mẫu Validation Fold 0.")
else:
    print("Cần GPU và dataset tại data/train để chạy.")

## 2. Phân tích Chồng chéo Lỗi 4 góc (4-way Error Overlap)

Chia các mẫu validation Fold 0 thành 4 nhóm:
1. Cả 2 cùng đúng
2. Chỉ Native đúng (B2 sai)
3. Chỉ B2 đúng (Native sai)
4. Cả 2 cùng sai

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Bước huấn luyện CNN cần CUDA. Bạn có thể chạy nhánh LR ở bài 01 và 05 trên CPU.")

if (data_root / 'pairs.csv').is_file() and torch.cuda.is_available():
    y_val = va_fold0.fake_position.to_numpy()
    p_nat = pred_nat.p.to_numpy()
    p_b2 = pred_b2.p.to_numpy()
    
    c_nat = (p_nat >= 0.5) == y_val
    c_b2 = (p_b2 >= 0.5) == y_val
    
    g_both_c = c_nat & c_b2
    g_only_nat = c_nat & (~c_b2)
    g_only_b2 = (~c_nat) & c_b2
    g_both_w = (~c_nat) & (~c_b2)
    
    print("=== PHÂN BỐ 4 NHÓM LỖI TRÊN VALIDATION FOLD 0 ===")
    print(f"1. Cả 2 cùng đúng:          {np.sum(g_both_c):3d} mẫu ({np.mean(g_both_c)*100:.1f}%)")
    print(f"2. Chỉ Native đúng (B2 sai): {np.sum(g_only_nat):3d} mẫu ({np.mean(g_only_nat)*100:.1f}%)")
    print(f"3. Chỉ B2 đúng (Native sai): {np.sum(g_only_b2):3d} mẫu ({np.mean(g_only_b2)*100:.1f}%)")
    print(f"4. Cả 2 cùng sai:           {np.sum(g_both_w):3d} mẫu ({np.mean(g_both_w)*100:.1f}%)")

## 3. Nhìn lại từng nhóm dự đoán

Chọn ID nhỏ nhất trong mỗi nhóm sau khi có dự đoán validation, hiển thị ảnh và xác suất của cả hai mô hình. Đây là phân tích hậu nghiệm để đặt câu hỏi; không dùng các ví dụ này làm chứng cứ cơ chế hoặc để chỉnh ngưỡng rồi gọi điểm số là độc lập.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_case(row, caption):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for side, ax in enumerate(axes):
        with Image.open(data_root / row[f'image_{side}']) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'image_{side}')
        ax.axis('off')
    fig.suptitle(f"{caption} | ID={row.pair_id} | ảnh giả={row.fake_position}")
    plt.tight_layout()
    plt.show()

for title, mask in [('Cả hai đúng', g_both_c), ('Chỉ Native đúng', g_only_nat),
                    ('Chỉ B2 đúng', g_only_b2), ('Cả hai sai', g_both_w)]:
    candidates = va_fold0.loc[mask].sort_values('pair_id')
    if candidates.empty:
        print(title, ': không có mẫu')
        continue
    row = candidates.iloc[0]
    i = va_fold0.index[va_fold0.pair_id == row.pair_id][0]
    show_case(row, f'{title}: p_native={p_nat[i]:.3f}, p_b2={p_b2[i]:.3f}')

## 4. Kết hợp Mô hình (Blending 50/50 B2 + Native)

Công thức kết hợp xác suất:
$$p_{\text{blend}} = 0.5 \cdot p_{\text{B2}} + 0.5 \cdot p_{\text{native}}$$

Chốt trọng số 50/50 và ngưỡng 0.5 trước khi xem kết quả blend. Dùng **B2 làm đối chiếu cố định**: “sửa lỗi” là B2 sai, blend đúng; “lỗi mới” là B2 đúng, blend sai. Hai mô hình có lỗi khác nhau chưa bảo đảm trung bình xác suất sẽ tốt hơn, vì độ tự tin cũng ảnh hưởng quyết định.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Bước huấn luyện CNN cần CUDA. Bạn có thể chạy nhánh LR ở bài 01 và 05 trên CPU.")

if (data_root / 'pairs.csv').is_file() and torch.cuda.is_available():
    pred_blend = blend(pred_b2, pred_nat, va_fold0)
    p_bl = pred_blend.p.to_numpy()
    c_bl = (p_bl >= 0.5) == y_val
    
    fixed_errs = (~c_b2) & c_bl
    new_errs = c_b2 & (~c_bl)
    
    s_nat = metric(y_val, p_nat)
    s_b2 = metric(y_val, p_b2)
    s_bl = metric(y_val, p_bl)
    
    df_blend_res = pd.DataFrame([
        {'Mô hình': '1. Native (DenseNet-121)', **s_nat},
        {'Mô hình': '2. B2 (EfficientNet-B2)', **s_b2},
        {'Mô hình': '3. Blend 50/50 (B2 + Native)', **s_bl},
    ])
    print("=== SO SÁNH HIỆU QUẢ BLENDING TRÊN FOLD 0 ===")
    print(df_blend_res[['Mô hình', 'macro_f1', 'accuracy', 'log_loss', 'errors']].to_string(index=False))
    print(f"\nSố lỗi được sửa nhờ Blend: {np.sum(fixed_errs)}")
    print(f"Số lỗi mới sinh ra do Blend: {np.sum(new_errs)}")

In [ ]:
assert int((~c_b2).sum() - fixed_errs.sum() + new_errs.sum()) == int((~c_bl).sum())
for title, mask in [('Blend sửa lỗi B2', fixed_errs), ('Blend tạo lỗi so với B2', new_errs)]:
    candidates = va_fold0.loc[mask].sort_values('pair_id')
    if candidates.empty:
        print(title, ': không có mẫu')
        continue
    row = candidates.iloc[0]
    i = va_fold0.index[va_fold0.pair_id == row.pair_id][0]
    show_case(row, f'{title}: p_b2={p_b2[i]:.3f}, p_native={p_nat[i]:.3f}, p_blend={p_bl[i]:.3f}')

## Kiểm tra lợi ích trước khi mở rộng

Blend sửa được nhiều lỗi hơn số lỗi mới không? Macro-F1 có cùng chiều với số lỗi không? So sánh cả hai vì Macro-F1 còn phụ thuộc phân bố lỗi giữa hai lớp. Nếu blend kém hơn, giữ nguyên kết quả đó; không thay trọng số để tạo một câu chuyện cải thiện.

Bài 05 hoàn tất các fold còn thiếu của phương pháp bạn chọn.